# 04 — Appendix A: LOO threshold sensitivity

Reproduces Appendix A of the manuscript: counts of holdouts in which at least one perturbation produces a leave-one-out impact above each of three thresholds, out of 100 CPA seeds per condition.

**Manuscript reference:** Appendix A; Methods §2.5 (threshold justification).

The appendix shows that the qualitative K562-vs-RPE1 contrast is preserved across adjacent thresholds (0.125, 0.150, 0.175), so the operational 0.150 single-driver flag is not finely tuned.

**Sources:**
- `precomputed/eval/diag_loo_sensitivity_n100.csv` (filtered to `model_id == "CPA"`).

**Outputs:**
- `precomputed/tables/appendix_a_threshold_sensitivity.csv`


In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from perturb_style import PRECOMPUTED_DIR


## Load data

In [2]:
cpa = pd.read_csv(PRECOMPUTED_DIR / "eval" / "diag_loo_sensitivity_n100.csv")
cpa = cpa[cpa.model_id == "CPA"].copy()
print(f"CPA rows: {len(cpa)}")
print(cpa.groupby(["cell_type", "split_type"]).size().to_string())


CPA rows: 400
cell_type  split_type
K562       random        100
           stratified    100
RPE1       random        100
           stratified    100


## Compute threshold counts

For each (cell_type, split_type) and each threshold, count the number of seeds where `LOO_max_abs_delta` exceeds the threshold.

In [3]:
THRESHOLDS = [0.125, 0.150, 0.175]
COLUMNS = [("K562", "random"), ("K562", "stratified"),
           ("RPE1", "random"), ("RPE1", "stratified")]

rows = []
for thr in THRESHOLDS:
    row = {"Threshold": f"LOO_max > {thr:.3f}"}
    for cell, split in COLUMNS:
        sub = cpa[(cpa.cell_type == cell) & (cpa.split_type == split)]
        n_above = int((sub.LOO_max_abs_delta > thr).sum())
        row[f"{cell} {split}"] = f"{n_above}/100"
    rows.append(row)

appendix_a = pd.DataFrame(rows)
print(appendix_a.to_string(index=False))


      Threshold K562 random K562 stratified RPE1 random RPE1 stratified
LOO_max > 0.125      69/100          65/100      21/100          20/100
LOO_max > 0.150      56/100          53/100      13/100           9/100
LOO_max > 0.175      49/100          44/100       6/100           2/100


## Save

In [4]:
out_dir = PRECOMPUTED_DIR / "tables"
out_dir.mkdir(exist_ok=True, parents=True)
out_path = out_dir / "appendix_a_threshold_sensitivity.csv"
appendix_a.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
appendix_a


Saved: precomputed/tables/appendix_a_threshold_sensitivity.csv


,Threshold,K562 random,K562 stratified,RPE1 random,RPE1 stratified
0,LOO_max > 0.125,69/100,65/100,21/100,20/100
1,LOO_max > 0.150,56/100,53/100,13/100,9/100
2,LOO_max > 0.175,49/100,44/100,6/100,2/100
